# U-Net like CNN with skip-connections (TRAIN)

## Imports
### Libs

In [ ]:
from contextlib import nullcontext
from pathlib import Path

import torch
from clearml import Dataset, Task
from torch import nn
from torch.utils.data import DataLoader

### Chromatica modules

In [ ]:
from chromatica.datasets.dataset import ImageDataset
from chromatica.nn.v2.cnn import CNN

### Check for CUDA or MPS

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is used")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is used")
else:
    device = torch.device("cpu")
    print("CPU is used")

## Train task init

In [ ]:
CLEARML_SAVE_TASK = True

task = Task.init(
    project_name="Chromatica",
    task_name="Train U-Net like with skip-connections",
)

logger = task.get_logger()

In [ ]:
params = {
    "learning_rate": 1e-4,
    "num_epochs": 1,
    "batch_size": 32,
    "num_workers": 6,
    "prefetch_factor": 4,
    "log_every_steps": 10,
    "ema_beta": 0.98,
}
params = task.connect_configuration(params)

## Load dataset

In [ ]:
food101_path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="Food101").get_local_copy()
)

coco_path = Path(
    Dataset.get(dataset_project="Colorization", dataset_name="COCO").get_local_copy()
)

In [ ]:
food101 = ImageDataset(food101_path / "train")
coco = ImageDataset(coco_path / "train")

In [ ]:
def loader(dataset):
    return DataLoader(
        dataset,
        batch_size=params["batch_size"],
        shuffle=True,
        num_workers=params["num_workers"],
        prefetch_factor=params["prefetch_factor"],
        persistent_workers=True,
        pin_memory=(device == torch.device("cuda")),
    )

## Train

In [ ]:
def mps_autocast_or_null():
    if device.type == "mps":
        try:
            return torch.autocast("mps", dtype=torch.float16)
        except Exception:
            return nullcontext()
    return nullcontext()

### Only `Food101`

In [ ]:
model = CNN().to(device)

In [ ]:
%%time

criterion = nn.MSELoss()
optim = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

model.train()

global_step = 0
ema_loss_t = None

for epoch in range(params["num_epochs"]):
    epoch_loss_t = torch.zeros((), device=device)
    num_batches = 0

    for batch_idx, (x_, y_, _) in enumerate(loader(food101)):
        x = x_.to(device, non_blocking=True)
        y = y_.to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)

        with mps_autocast_or_null():
            pred = model(x)

        loss = criterion(pred, y)

        loss.backward()
        optim.step()

        num_batches += 1
        global_step += 1
        loss_det = loss.detach()
        epoch_loss_t += loss_det

        if ema_loss_t is None:
            ema_loss_t = loss_det
        else:
            ema_loss_t = (
                params["ema_beta"] * ema_loss_t + (1.0 - params["ema_beta"]) * loss_det
            )

        if (batch_idx + 1) % params["log_every_steps"] == 0 or batch_idx == 0:
            logger.report_scalar(
                "Food101 Loss",
                "train_batch",
                float(loss_det.item()),
                iteration=global_step,
            )
            logger.report_scalar(
                "Food101 Loss",
                "train_batch_ema",
                float(ema_loss_t.item()),
                iteration=global_step,
            )

    mean_epoch_loss = float((epoch_loss_t / num_batches).item())
    logger.report_scalar(
        "Food101 Loss", "train_epoch", mean_epoch_loss, iteration=epoch
    )

In [ ]:
model_path = Path("./.data")
model_path.mkdir(exist_ok=True, parents=True)
torch.save(model.state_dict(), model_path / "nn_v2_food101.pth")
task.upload_artifact("model_food101", model_path / "nn_v2_food101.pth")

### Only `COCO`

In [ ]:
model = CNN().to(device)

In [ ]:
%%time

criterion = nn.MSELoss()
optim = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

model.train()

global_step = 0
ema_loss_t = None

for epoch in range(params["num_epochs"]):
    epoch_loss_t = torch.zeros((), device=device)
    num_batches = 0

    for batch_idx, (x_, y_, _) in enumerate(loader(coco)):
        x = x_.to(device, non_blocking=True)
        y = y_.to(device, non_blocking=True)

        optim.zero_grad(set_to_none=True)

        with mps_autocast_or_null():
            pred = model(x)

        loss = criterion(pred, y)

        loss.backward()
        optim.step()

        num_batches += 1
        global_step += 1
        loss_det = loss.detach()
        epoch_loss_t += loss_det

        if ema_loss_t is None:
            ema_loss_t = loss_det
        else:
            ema_loss_t = (
                params["ema_beta"] * ema_loss_t + (1.0 - params["ema_beta"]) * loss_det
            )

        if (batch_idx + 1) % params["log_every_steps"] == 0 or batch_idx == 0:
            logger.report_scalar(
                "COCO Loss",
                "train_batch",
                float(loss_det.item()),
                iteration=global_step,
            )
            logger.report_scalar(
                "COCO Loss",
                "train_batch_ema",
                float(ema_loss_t.item()),
                iteration=global_step,
            )

    mean_epoch_loss = float((epoch_loss_t / num_batches).item())
    logger.report_scalar("COCO Loss", "train_epoch", mean_epoch_loss, iteration=epoch)

In [ ]:
model_path = Path("./.data")
model_path.mkdir(exist_ok=True, parents=True)
torch.save(model.state_dict(), model_path / "nn_v2_coco.pth")
task.upload_artifact("model_coco", model_path / "nn_v2_coco.pth")

In [ ]:
if CLEARML_SAVE_TASK:
    task.mark_completed()
else:
    task.close()
    task.set_archived(True)